In [1]:
# Client Setup
import boto3

client = boto3.client("bedrock-runtime", region_name="eu-central-1")
model_id = "eu.anthropic.claude-sonnet-4-5-20250929-v1:0"

In [2]:
# Helper functions


def add_user_message(messages, content):
    if isinstance(content, str):
        user_message = {"role": "user", "content": [{"text": content}]}
    else:
        user_message = {"role": "user", "content": content}
    messages.append(user_message)


def add_assistant_message(messages, content):
    if isinstance(content, str):
        assistant_message = {
            "role": "assistant",
            "content": [{"text": content}],
        }
    else:
        assistant_message = {"role": "assistant", "content": content}

    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice="auto",
):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences,
        },
    }

    if system:
        params["system"] = [{"text": system}]

    tool_choices = {
        "auto": {"auto": {}},
        "any": {"any": {}},
    }
    if tools:
        choice = tool_choices.get(tool_choice, {"tool": {"name": tool_choice}})
        params["toolConfig"] = {"tools": tools, "toolChoice": choice}

    response = client.converse(**params)
    parts = response["output"]["message"]["content"]

    return {
        "parts": parts,
        "stop_reason": response["stopReason"],
        "text": "\n".join([p["text"] for p in parts if "text" in p]),
    }

In [3]:
# Tool Schemas

article_details_schema = {
    "toolSpec": {
        "name": "article_details",
        "description": "This tool should be called with details about an article. It accepts information about the article's title, author, and related topics.",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "The title of the article. Can be left empty.",
                    },
                    "author": {
                        "type": "string",
                        "description": "The name of the article's author. Can be left empty.",
                    },
                    "topics": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "A list of topics or categories that the article covers. Can be an empty list.",
                    },
                },
            }
        },
    }
}

to_json_schema = {
    "toolSpec": {
        "name": "to_json",
        "description": "This tool processes any JSON data and can be used for generating structured content, transforming information, or creating any JSON-based output needed for your task.",
        "inputSchema": {
            "json": {"type": "object", "additionalProperties": True}
        },
    }
}

In [4]:
messages = []

add_user_message(
    messages,
    "Write a one-paragraph scholarly article about computer science. Include a title and author name",
)

result = chat(messages)

add_assistant_message(messages, result["text"])

result["text"]

"**Quantum Computing and the Future of Cryptographic Security**\n\n*Dr. Sarah Chen, Department of Computer Science, MIT*\n\nThe advent of quantum computing represents both a revolutionary advancement and an existential threat to contemporary cryptographic systems, necessitating immediate attention from the computer science community. Current public-key cryptography standards, including RSA and elliptic curve cryptography (ECC), derive their security from the computational intractability of problems such as integer factorization and discrete logarithms on classical computers; however, Shor's algorithm, executable on sufficiently powerful quantum computers, can solve these problems in polynomial time, thereby rendering such encryption schemes vulnerable. As major technology corporations and research institutions make steady progress toward achieving quantum supremacy—the point at which quantum computers can perform calculations beyond the reach of classical machines—the development and i

In [5]:
messages = []

add_user_message(messages, f"""
Analyze the article below and extract key data. Then call the article_details tool.

<article_text>
{result["text"]}
</article_text>
""")

json_result = chat(messages, tools=[article_details_schema], tool_choice="article_details")

json_result

{'parts': [{'toolUse': {'toolUseId': 'tooluse_U1o3BW0es6dPQesqJn8bky',
    'name': 'article_details',
    'input': {'title': 'Quantum Computing and the Future of Cryptographic Security',
     'author': 'Dr. Sarah Chen',
     'topics': ['quantum computing',
      'cryptography',
      'cryptographic security',
      'post-quantum cryptography',
      'RSA',
      'elliptic curve cryptography',
      "Shor's algorithm",
      'quantum supremacy',
      'NIST',
      'lattice-based cryptography',
      'hash-based cryptography',
      'code-based cryptography',
      'multivariate polynomial cryptography',
      'cybersecurity']},
    'type': 'tool_use'}}],
 'stop_reason': 'tool_use',
 'text': ''}

In [6]:
messages = []

add_user_message(messages, f"""
Analyze the article below and extract key data. Then call the to_json tool.

<article_text>
{result["text"]}
</article_text>

WHen you call to_json, pass in the following structure:
{{
    "title": str # title of the article,
    "author": str # author of the article,
    "topics": List[str] # List of topics mentioned in the article
}}
""")

flexible_result = chat(messages, tools=[to_json_schema], tool_choice="to_json")

flexible_result

{'parts': [{'toolUse': {'toolUseId': 'tooluse_lP8VMRoIw4fa94UXfFbcwW',
    'name': 'to_json',
    'input': {'title': 'Quantum Computing and the Future of Cryptographic Security',
     'author': 'Dr. Sarah Chen, Department of Computer Science, MIT',
     'topics': ['quantum computing',
      'cryptographic security',
      'public-key cryptography',
      'RSA',
      'elliptic curve cryptography (ECC)',
      'integer factorization',
      'discrete logarithms',
      "Shor's algorithm",
      'quantum supremacy',
      'post-quantum cryptographic algorithms',
      'National Institute of Standards and Technology (NIST)',
      'lattice-based cryptography',
      'hash-based cryptography',
      'code-based cryptography',
      'multivariate polynomial cryptography',
      'hybrid cryptographic systems',
      'quantum-resistant infrastructure']},
    'type': 'tool_use'}}],
 'stop_reason': 'tool_use',
 'text': ''}